
# Pediatric Mental Health Access: Full Project Analysis

## Project Focus

This notebook combines the final statistical analyses from all datasets used in the pediatric mental health access project.

### Primary Research Question

Among school-aged children ages 6–17 with current anxiety, depression, or ADHD, what factors are most strongly associated with unmet mental health care, and which communities are at greatest risk?

### Secondary Research Question

Among those associated factors, which are potentially modifiable and present the strongest opportunities for early digital or service intervention?

## Dataset Stack

- National Survey of Children's Health (NSCH), 2024
- Youth Risk Behavior Surveillance System (YRBSS), 2023
- American Community Survey (ACS), 2024 5-year
- NCES Common Core of Data Fiscal
- NCES Common Core of Data Nonfiscal
- HRSA Mental Health Professional Shortage Areas (HPSA)

## Analysis Strategy

Each dataset contributes a different layer of evidence:

- **NSCH:** child and family mental health need, care access, and risk factors
- **YRBSS:** adolescent mental health experiences and behavioral risk
- **ACS:** socioeconomic and community context
- **NCES Fiscal:** district financial resources
- **NCES Nonfiscal:** district staffing and school-system characteristics
- **HPSA:** mental health provider shortages and geographic access

The final section synthesizes findings across datasets to identify population risk, structural barriers, and potentially modifiable intervention opportunities.



# 1. Setup


In [ ]:

from pathlib import Path

import pandas as pd
import numpy as np

from scipy.stats import chi2_contingency
from sklearn.linear_model import LogisticRegression

project_root = Path.cwd().parents[1]

cleaned_dir = project_root / "data" / "cleaned"

print("Project root:", project_root)
print("Cleaned data folder:", cleaned_dir)



# 2. National Survey of Children's Health (NSCH)

## Research Role

NSCH provides the child- and family-level foundation of the project.

The primary NSCH analysis examines:

- current anxiety, depression, and ADHD
- age patterns
- comorbidity
- mental health care need
- unmet mental health care
- family, socioeconomic, school, insurance, ACE, neighborhood, and demographic factors associated with unmet care



## 2.1 Load NSCH Data


In [ ]:

nsch_path = cleaned_dir / "NSCH_2024_cleaned_ages_6_to_17.csv"

nsch = pd.read_csv(nsch_path)

print("NSCH shape:", nsch.shape)



## 2.2 Condition Prevalence


In [ ]:

current_conditions = {
    "Anxiety": "anxiety_24",
    "Depression": "depress_24",
    "ADHD": "ADHD_24"
}

prevalence_results = []

for label, col in current_conditions.items():

    valid = nsch[nsch[col] != 99]

    current = (valid[col] == 3).sum()
    total = len(valid)

    prevalence_results.append({
        "Condition": label,
        "Current cases": current,
        "Valid responses": total,
        "Percent": round(current / total * 100, 1)
    })

nsch_prevalence = pd.DataFrame(prevalence_results)

nsch_prevalence



## 2.3 Age Groups


In [ ]:

nsch["age_group"] = pd.cut(
    nsch["child_age_years"],
    bins=[5, 11, 17],
    labels=["Ages 6–11", "Ages 12–17"]
)

age_group_results = []

for age_group in ["Ages 6–11", "Ages 12–17"]:

    group = nsch[nsch["age_group"] == age_group]

    for label, col in current_conditions.items():

        valid = group[group[col] != 99]

        current = (valid[col] == 3).sum()
        total = len(valid)

        age_group_results.append({
            "Age group": age_group,
            "Condition": label,
            "Current cases": current,
            "Valid responses": total,
            "Percent": round(current / total * 100, 1)
        })

nsch_age_groups = pd.DataFrame(age_group_results)

nsch_age_groups



## 2.4 Target Mental Health Population


In [ ]:

valid_condition_data = nsch[
    (nsch["anxiety_24"] != 99) &
    (nsch["depress_24"] != 99) &
    (nsch["ADHD_24"] != 99)
].copy()

valid_condition_data["current_condition_count"] = (
    (valid_condition_data["anxiety_24"] == 3).astype(int) +
    (valid_condition_data["depress_24"] == 3).astype(int) +
    (valid_condition_data["ADHD_24"] == 3).astype(int)
)

target_population = valid_condition_data[
    valid_condition_data["current_condition_count"] >= 1
].copy()

print("Target population:", f"{len(target_population):,}")
print(
    "Percent of valid sample:",
    f"{len(target_population) / len(valid_condition_data) * 100:.1f}%"
)



## 2.5 Comorbidity


In [ ]:

target_population["condition_combination"] = target_population.apply(
    lambda row: " + ".join([
        condition
        for condition, present in [
            ("Anxiety", row["anxiety_24"] == 3),
            ("Depression", row["depress_24"] == 3),
            ("ADHD", row["ADHD_24"] == 3)
        ]
        if present
    ]),
    axis=1
)

condition_combinations = (
    target_population["condition_combination"]
    .value_counts()
    .reset_index()
)

condition_combinations.columns = [
    "Condition combination",
    "Children"
]

condition_combinations["Percent"] = (
    condition_combinations["Children"]
    / len(target_population) * 100
).round(1)

condition_combinations



## 2.6 Mental Health Care Access


In [ ]:

care_population = target_population[
    target_population["MentHCare_24"].isin([1, 2])
].copy()

care_population["unmet_care"] = (
    care_population["MentHCare_24"] == 2
).astype(int)

print("Children who needed care:", len(care_population))
print("Children with unmet care:", care_population["unmet_care"].sum())
print(
    "Unmet care rate:",
    f"{care_population['unmet_care'].mean() * 100:.1f}%"
)



## 2.7 Chi-Square Tests and Cramér's V


In [ ]:

care_population["condition_burden"] = (
    care_population["current_condition_count"]
    .map({
        1: "One condition",
        2: "Two conditions",
        3: "Three conditions"
    })
)

chi_square_factors = [
    "age_group",
    "condition_burden",
    "condition_combination",
    "povlev4_24",
    "CurrIns_24",
    "InsGap_24",
    "InsAdeq_24",
    "ACE2more_24",
    "ParentMH_24",
    "FAMILY_R",
    "HousingInstab_24",
    "SchlEngage_24",
    "SchlSafe_24",
    "neighborhood_support",
    "neighborhood_safety",
    "child_sex",
    "child_race",
    "child_hispanic_ethnicity"
]

chi_square_results = []

for factor in chi_square_factors:

    test_data = care_population[
        care_population[factor].notna()
    ].copy()

    if pd.api.types.is_numeric_dtype(test_data[factor]):

        test_data = test_data[
            ~test_data[factor].isin([90, 95, 99])
        ]

    contingency = pd.crosstab(
        test_data[factor],
        test_data["unmet_care"]
    )

    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        continue

    chi2, p_value, dof, expected = chi2_contingency(
        contingency
    )

    n = contingency.to_numpy().sum()

    min_dimension = min(
        contingency.shape[0] - 1,
        contingency.shape[1] - 1
    )

    cramers_v = np.sqrt(
        chi2 / (n * min_dimension)
    )

    chi_square_results.append({
        "Factor": factor,
        "N": n,
        "Chi-square": round(chi2, 2),
        "Degrees of freedom": dof,
        "p-value": p_value,
        "Cramer's V": round(cramers_v, 3)
    })

nsch_chi_square = (
    pd.DataFrame(chi_square_results)
    .sort_values("Cramer's V", ascending=False)
    .reset_index(drop=True)
)

nsch_chi_square



## 2.8 Logistic Regression


In [ ]:

model_vars = [
    "unmet_care",
    "child_age_years",
    "current_condition_count",
    "povlev4_24",
    "InsAdeq_24",
    "ACE2more_24",
    "ParentMH_24",
    "FAMILY_R",
    "SchlEngage_24",
    "neighborhood_support"
]

model_data = care_population[
    model_vars
].copy()

for col in model_vars[3:]:

    model_data = model_data[
        ~model_data[col].isin([90, 95, 99])
    ]

model_data = model_data.dropna()

X = model_data.drop(columns="unmet_care")
y = model_data["unmet_care"]

categorical_vars = [
    "povlev4_24",
    "InsAdeq_24",
    "ACE2more_24",
    "ParentMH_24",
    "FAMILY_R",
    "SchlEngage_24",
    "neighborhood_support"
]

X = pd.get_dummies(
    X,
    columns=categorical_vars,
    drop_first=True,
    dtype=int
)

log_reg = LogisticRegression(
    max_iter=3000
)

log_reg.fit(X, y)

nsch_regression = pd.DataFrame({
    "Predictor": X.columns,
    "Coefficient": log_reg.coef_[0],
    "Odds Ratio": np.exp(log_reg.coef_[0])
})

nsch_regression["Distance from 1"] = (
    nsch_regression["Odds Ratio"] - 1
).abs()

nsch_regression = (
    nsch_regression
    .sort_values("Distance from 1", ascending=False)
    .reset_index(drop=True)
)

nsch_regression[
    ["Predictor", "Coefficient", "Odds Ratio"]
].round(3)



## 2.9 Exploratory Data Analysis Visualizations

The following visualizations summarize the major distributions and relationships identified during the NSCH analysis. These plots focus on mental health prevalence, age differences, comorbidity, and factors associated with unmet mental health care.


In [ ]:

import matplotlib.pyplot as plt

condition_plot = pd.DataFrame({
    "Condition": ["Anxiety", "Depression", "ADHD"],
    "Percent": [17.4, 6.8, 16.3]
})

plt.figure(figsize=(8, 5))

bars = plt.bar(
    condition_plot["Condition"],
    condition_plot["Percent"]
)

plt.title("Current Mental Health Conditions Among Children Ages 6–17")
plt.ylabel("Percent of Valid Responses")
plt.xlabel("")

for bar, value in zip(bars, condition_plot["Percent"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.4,
        f"{value:.1f}%",
        ha="center"
    )

plt.ylim(0, 22)
plt.tight_layout()
plt.show()



### Finding: Current Condition Prevalence

Among the three conditions included in the target population, current anxiety was the most common at 17.4%, followed closely by ADHD at 16.3%. Current depression was reported for 6.8% of valid responses.

These unweighted percentages describe the cleaned NSCH sample and should not be interpreted as national prevalence estimates.


In [ ]:

age_condition_plot = pd.DataFrame({
    "Condition": [
        "Anxiety", "Depression", "ADHD",
        "Anxiety", "Depression", "ADHD"
    ],
    "Age Group": [
        "Ages 6–11", "Ages 6–11", "Ages 6–11",
        "Ages 12–17", "Ages 12–17", "Ages 12–17"
    ],
    "Percent": [
        12.9, 2.4, 14.9,
        20.5, 10.0, 17.3
    ]
})

pivot_age = age_condition_plot.pivot(
    index="Condition",
    columns="Age Group",
    values="Percent"
)

ax = pivot_age.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title("Current Mental Health Conditions by Age Group")
plt.ylabel("Percent of Valid Responses")
plt.xlabel("")
plt.xticks(rotation=0)
plt.legend(title="Age Group")

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3
    )

plt.tight_layout()
plt.show()



### Finding: Mental Health Conditions by Age Group

Among children ages 6–11, ADHD was the most common of the three conditions at 14.9%, compared with 12.9% for anxiety and 2.4% for depression.

Among adolescents ages 12–17, anxiety increased to 20.5% and became the most common condition. Depression also increased substantially to 10.0%, while ADHD increased more modestly to 17.3%.

Age is therefore particularly important for understanding the distribution of anxiety and depression, even though later analyses found age to have a relatively weak association with unmet care.


In [ ]:

comorbidity_plot = (
    target_population["current_condition_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)

comorbidity_plot.columns = [
    "Number of Conditions",
    "Children"
]

comorbidity_plot["Percent"] = (
    comorbidity_plot["Children"]
    / comorbidity_plot["Children"].sum()
    * 100
)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    comorbidity_plot["Number of Conditions"].astype(str),
    comorbidity_plot["Percent"]
)

plt.title("Number of Current Conditions Within the Mental Health Population")
plt.xlabel("Number of Current Conditions")
plt.ylabel("Percent of Target Population")

for bar, value in zip(
    bars,
    comorbidity_plot["Percent"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f"{value:.1f}%",
        ha="center"
    )

plt.ylim(0, 70)
plt.tight_layout()
plt.show()



### Finding: Comorbidity

Comorbidity was common among children with current anxiety, depression, or ADHD.

Approximately 59.3% had one of the three conditions, 29.6% had two, and 11.1% had all three.

Together, 40.7% of the target mental health population had two or more current conditions.


In [ ]:

top_associations = (
    nsch_chi_square
    .sort_values("Cramer's V", ascending=False)
    .head(8)
    .sort_values("Cramer's V")
)

plt.figure(figsize=(9, 6))

plt.barh(
    top_associations["Factor"],
    top_associations["Cramer's V"]
)

plt.title("Strongest Associations With Unmet Mental Health Care")
plt.xlabel("Cramér's V")
plt.ylabel("")

plt.tight_layout()
plt.show()



### Finding: Strongest Associations With Unmet Care

The strongest bivariate associations with unmet mental health care were concentrated in family, socioeconomic, insurance, school, and neighborhood conditions.

Parent mental health showed the largest Cramér's V, followed by poverty level, family factors, neighborhood support, ACE exposure, insurance gaps, neighborhood safety, and school engagement.

Although these relationships were statistically significant, effect sizes were generally small, suggesting that unmet mental health care is associated with multiple overlapping conditions rather than one dominant risk factor.



# 3. American Community Survey (ACS)

## Research Role

ACS provides socioeconomic, demographic, household, and community context.

This section will examine which state and district characteristics overlap with the risk factors identified in NSCH.

**Final ACS analysis code will be inserted here after `02_acs_data_analysis.ipynb` is complete.**


In [ ]:

acs_path = cleaned_dir / "ACS_2024_cleaned_state_district.csv"

acs = pd.read_csv(acs_path)

print("ACS shape:", acs.shape)



# 4. NCES Fiscal

## Research Role

NCES fiscal data provides district-level resource and funding context.

**Final NCES fiscal analysis code will be inserted here after `03_nces_fiscal_data_analysis.ipynb` is complete.**


In [ ]:

nces_fiscal_path = cleaned_dir / "NCES_CCD_2022_23_Fiscal_cleaned.csv"

nces_fiscal = pd.read_csv(nces_fiscal_path)

print("NCES fiscal shape:", nces_fiscal.shape)



# 5. NCES Nonfiscal

## Research Role

NCES nonfiscal data provides district staffing, enrollment, and school-system context.

**Final NCES nonfiscal analysis code will be inserted here after `04_nces_nonfiscal_data_analysis.ipynb` is complete.**


In [ ]:

nces_directory_path = (
    cleaned_dir /
    "NCES_CCD_2024_25_District_Directory_clean.csv"
)

nces_staff_path = (
    cleaned_dir /
    "NCES_CCD_2024_25_District_Staff_clean.csv"
)

nces_directory = pd.read_csv(nces_directory_path)
nces_staff = pd.read_csv(nces_staff_path)

print("Directory shape:", nces_directory.shape)
print("Staff shape:", nces_staff.shape)



# 6. HRSA Mental Health HPSA

## Research Role

HPSA data provides geographic context on mental health professional shortages and provider access.

**Final HPSA analysis code will be inserted here after `05_hpsa_data_analysis.ipynb` is complete.**


In [ ]:

hpsa_path = (
    cleaned_dir /
    "HPSA_20206_snapshot_mental_health_cleaned.csv"
)

hpsa = pd.read_csv(hpsa_path)

print("HPSA shape:", hpsa.shape)



# 7. Youth Risk Behavior Surveillance System (YRBSS)

## Research Role

YRBSS provides adolescent behavioral-health and school-age youth context, complementing parent-reported NSCH measures.

**Final YRBSS analysis code will be inserted here after `06_yrbss_data_analysis.ipynb` is complete.**


In [ ]:

yrbss_path = cleaned_dir / "yrbss_2023_cleaned.csv"

yrbss = pd.read_csv(yrbss_path)

print("YRBSS shape:", yrbss.shape)



# 8. Cross-Dataset Synthesis

This section will combine the strongest findings from all six datasets.

The synthesis will answer:

1. What individual and family factors are associated with unmet mental health care?
2. Which communities show overlapping socioeconomic, school-resource, and provider-access risks?
3. Which findings are consistent across datasets?
4. Where do datasets provide different but complementary evidence?
5. Which factors appear modifiable through digital, service, school, or navigation interventions?
6. Which findings should lead the final research report and dashboards?



## Final Evidence Table

A final synthesis table will be created with fields such as:

| Finding | Dataset | Evidence | Strength | Geographic Level | Modifiable? | Intervention Relevance |
|---|---|---|---|---|---|---|

This table will become the bridge between statistical analysis and the Project 4 research report.
